In [33]:
import pandas as pd
df = pd.read_csv("data/ads.csv")
df.columns

Index(['id', 'ad_creation_time', 'ad_delivery_start_time', 'bylines',
       'currency', 'page_id', 'page_name', 'ad_creative_bodies',
       'ad_creative_link_captions', 'ad_creative_link_descriptions',
       'ad_creative_link_titles', 'audience_size_lower', 'impressions_lower',
       'impressions_upper', 'spend_lower', 'spend_upper', 'ad_snapshot_url',
       'language_en', 'language_es', 'language_other', 'platform_fb',
       'platform_ig', 'platform_other', '13-17/female', '13-17/male',
       '13-17/unknown', '18-24/female', '18-24/male', '18-24/unknown',
       '25-34/female', '25-34/male', '25-34/unknown', '35-44/female',
       '35-44/male', '35-44/unknown', '45-54/female', '45-54/male',
       '45-54/unknown', '55-64/female', '55-64/male', '55-64/unknown',
       '65+/female', '65+/male', '65+/unknown', 'Unknown/female',
       'Unknown/male', 'Unknown/unknown'],
      dtype='object')

In [34]:
buzz_df = df.copy()

buzz_df['year'] = pd.to_datetime(buzz_df['ad_creation_time']).dt.year
buzz_df = buzz_df[buzz_df['ad_creative_bodies'].notna()]

buzz_df['clean_text'] = buzz_df['ad_creative_bodies'].str.lower()

demo_cols = [col for col in buzz_df.columns if '/' in col]
demo_cols = [col for col in demo_cols if not col.startswith('Unknown/')]

age_ranges = sorted(set(col.split('/')[0] for col in demo_cols))
genders = sorted(set(col.split('/')[1] for col in demo_cols))

# age totals
for age in age_ranges:
    age_specific_cols = [col for col in demo_cols if col.startswith(age + '/')]
    buzz_df[f'age_{age}_total'] = buzz_df[age_specific_cols].sum(axis=1)

age_total_cols = [f'age_{age}_total' for age in age_ranges]

# gender totals
for gender in genders:
    gender_specific_cols = [col for col in demo_cols if col.endswith('/' + gender)]
    buzz_df[f'gender_{gender}_total'] = buzz_df[gender_specific_cols].sum(axis=1)

gender_total_cols = [f'gender_{gender}_total' for gender in genders]

# original gender + age totals
age_gender_cols = demo_cols.copy()

buzz_df.head()

,id,ad_creation_time,ad_delivery_start_time,bylines,currency,page_id,page_name,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,...,age_13-17_total,age_18-24_total,age_25-34_total,age_35-44_total,age_45-54_total,age_55-64_total,age_65+_total,gender_female_total,gender_male_total,gender_unknown_total
0,1263237120532732,2020-04-01,2020-04-01,The Unreported Story Society,USD,455147011996099,The Ann and Phelim Scoop,Cancel Culture is killing intellectual convers...,youtu.be,NaN,...,0.0,54.17,33.49,5.78,2.62,2.20,1.75,57.79,41.85,0.37
1,2508232389430627,2020-04-01,2020-04-01,The Unreported Story Society,USD,455147011996099,The Ann and Phelim Scoop,On this week's episode we are joined by Bret S...,youtu.be,NaN,...,0.0,44.80,31.17,5.07,4.60,8.06,6.31,51.23,48.40,0.38
2,156182708989204,2020-04-01,2020-04-01,"Knowhere, Inc.",USD,950998258312654,Knowhere,Straight facts to see through the hysteria.We ...,fb.me,"Get our free, daily newsletter",...,0.0,0.35,1.29,5.28,20.94,39.23,32.91,48.63,50.71,0.66
3,397644197860634,2020-04-01,2020-04-01,"Knowhere, Inc.",USD,950998258312654,Knowhere,Straight facts to see through the hysteria.We ...,fb.me,"Get our free, daily newsletter",...,0.0,0.34,1.41,6.05,23.08,39.91,29.20,48.17,51.18,0.64
4,527071631288361,2020-04-01,2020-04-01,"Knowhere, Inc.",USD,950998258312654,Knowhere,Straight facts to see through the hysteria.We ...,fb.me,"Get our free, daily newsletter",...,0.0,0.35,1.58,7.43,25.35,40.20,25.08,43.19,56.14,0.66


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import matplotlib.pyplot as plt

In [36]:

global_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, min_df=5)
X_global = global_vectorizer.fit_transform(buzz_df['clean_text'])
feature_names_global = np.array(global_vectorizer.get_feature_names_out())
tfidf_means_global = np.asarray(X_global.mean(axis=0)).flatten()
top_100_indices = tfidf_means_global.argsort()[-100:][::-1]
top_100_buzzwords = feature_names_global[top_100_indices]

In [37]:
buzz_df['dominant_age'] = buzz_df[age_total_cols].idxmax(axis=1)
buzz_df['dominant_age'] = buzz_df['dominant_age'].str.replace('age_', '').str.replace('_total', '')

### chi-square test of homogeneity

null hypothesis: the distribution of buzzwords is the same across age groups

alt hypothesis: the distribution of buzzwords differs acrossage groups (evidence of micro-targeting)



In [ ]:
from scipy import stats

results = []
bonferroni_alpha = 0.05 / len(top_100_buzzwords)

for word in top_100_buzzwords:
    buzz_df['has_word'] = buzz_df['clean_text'].str.contains(r'\b' + word + r'\b', regex=True)
    contingency = pd.crosstab(buzz_df['dominant_age'], buzz_df['has_word'])

    T_stat, p_value, dof, expected_freq = stats.chi2_contingency(contingency)

    results.append({
        'buzzword': word,
        'T_stat': round(T_stat, 4),
        'p_value': p_value,
        'dof': dof,
        'significant': p_value < bonferroni_alpha
    })

results_df = pd.DataFrame(results).sort_values('p_value')
sig_words = results_df[results_df['significant'] == True]



In [ ]:
for i, row in results_df.iterrows():
    if row['significant']:
        print(f"For buzzword '{row['buzzword']}', we reject the null hypothesis that its distribution is the same across age groups. (T={row['T_stat']}, p={row['p_value']:.4e})")
    else:
        print(f"For buzzword '{row['buzzword']}', we fail to reject the null hypothesis that its distribution is the same across age groups.")

In [ ]:
top_sig = sig_words.head(30)

plt.figure(figsize=(10, 6))
plt.barh(top_sig['buzzword'], top_sig['T_stat'], color='maroon')
plt.xlabel('Chi-Square Statistic')
plt.title('Top 30 Buzzwords with Unequal Distribution Across Age Groups')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()